# Solar Filament Segmentation 2026 -- end-to-end inference pipeline

Instance segmentation of solar filaments in H-alpha images (MAGFiLO). The notebook runs either of two
pipelines, chosen by `STAGE_A` in the configuration cell.

**`STAGE_A = "effv2s"` (default) -- the final submission, public leaderboard 0.38** (`filament_effv2s.py`)

1. **Stage-A** -- five U-Nets with an EfficientNetV2-S encoder (one per fold). Each sees the full
   2048 x 2048 image, normalised by the mean / SD inside a 950 px disk, in four flip / rotation views;
   the five maps are averaged.
2. **Instances** -- 8-connected regions of `prob > 0.35` with a peak above 0.5, `area >= 100` and
   `mean prob >= 0.5`. There is no refiner.
3. **Near-limb filter** -- instances whose centroid lies beyond 0.88 of the solar-disk radius (disk
   from an Otsu threshold) are dropped.
4. **Instance scorer** -- a gradient-boosted regressor (`models/effv2s_scorer.txt`, fitted on the 6,460
   out-of-fold instances of these models) predicts every instance's expected matched IoU from
   inference-time features; instances below the PQ break-even (tau = 0.1885) are dropped.
5. One mask = one row (`filament_id`, compressed COCO RLE `counts`).

Reported: out-of-fold PQ 0.4318 (657 images), hold-out PQ 0.4309 (50 images), public leaderboard 0.38.

**`STAGE_A = "resnet34"` -- the earlier pipeline, public leaderboard 0.37** (`filament_pipeline.py`)

5-fold ResNet34 U-Net ensemble with dihedral test-time augmentation on 512 px sliding windows;
candidates `prob > 0.4`, 3x3 closing, `400 <= area <= 40000`, `mean prob >= 0.7`; every candidate
redrawn by a crop refiner (`smp.Unet(resnet34)`, input `[crop, CLAHE(crop), seed]`); the same
near-limb filter; scorer `models/instance_scorer.txt` (tau = 0.1972; `TAU = 0.15` is the balanced
Dice/PQ configuration). Reported: hold-out PQ 0.4427, out-of-fold 0.4089, public leaderboard 0.37.

Weights (Apache-2.0): Kaggle dataset
[`abukayyisahbasyid/solar-filament-segmentation-2026-weights`](https://www.kaggle.com/datasets/abukayyisahbasyid/solar-filament-segmentation-2026-weights)
(`effv2s_fold1..5.pt`, `unet_fold1..5.pt`, `refiner_best.pt`; SHA256 checksums in the repository's
`SHA256SUMS.txt`). No training happens in this notebook; training scripts are in `training/`.

## 1. Setup and configuration

In [ ]:
import subprocess, sys
try:
    import segmentation_models_pytorch  # noqa: F401
except ImportError:                     # not preinstalled on Kaggle images
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "segmentation-models-pytorch==0.5.0", "timm==1.0.26"], check=True)

In [ ]:
import os, sys, glob, json, time
import numpy as np, pandas as pd

sys.path.insert(0, ".")          # run from the folder that contains filament_pipeline.py
import filament_pipeline as fp
import filament_effv2s as fe
import filament_metric as fm
import filament_scorer as fs

# --- knobs -------------------------------------------------------------------------
STAGE_A = "effv2s"          # "effv2s": the final submission (public 0.38); "resnet34": the earlier pipeline (public 0.37)
MAX_TEST_IMAGES = None      # None = all 180 test images
EVAL_HOLDOUT_IMAGES = 0     # >0: also score that many hold-out images with the official PQ
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
assert STAGE_A in ("effv2s", "resnet34")

def find_first(root, predicate):
    for r, _, files in os.walk(root):
        for f in files:
            if predicate(r, f):
                return os.path.join(r, f)

def find_weights(pattern):
    return sorted(glob.glob(os.path.join(WEIGHTS_DIR, pattern))) if WEIGHTS_DIR else sorted(
        glob.glob(os.path.join(INPUT, "**", pattern), recursive=True))

# --- locate weights and competition data (Kaggle layout, or set the env vars) ------
INPUT = os.environ.get("FILAMENT_INPUT", "/kaggle/input")
WEIGHTS_DIR = os.environ.get("FILAMENT_WEIGHTS")
fold_paths = find_weights("effv2s_fold*.pt" if STAGE_A == "effv2s" else "unet_fold*.pt")
refiner_path = (find_weights("refiner_best.pt") or [None])[0] if STAGE_A == "resnet34" else None
ann_json = find_first(INPUT, lambda r, f: f.endswith(".json") and "nnotation" in f)
train_dir = next((r for r, _, _ in os.walk(INPUT) if os.path.basename(r) == "train_images"), None)
test_dir = next((r for r, _, _ in os.walk(INPUT) if os.path.basename(r) == "test_images"), None)
assert len(fold_paths) == 5 and (refiner_path or STAGE_A == "effv2s"), "weights not found (attach the weights dataset)"
print(STAGE_A, "|", len(fold_paths), "stage-A checkpoints |", refiner_path or "no refiner")
print("annotations:", ann_json, "| train:", train_dir, "| test:", test_dir)

## 2. Load models

In [ ]:
import torch
device = fp.default_device()
TAU = None   # None: the stored scorer threshold (effv2s 0.1885; resnet34 0.1972). resnet34 only: 0.15 = the balanced Dice/PQ configuration
if STAGE_A == "effv2s":
    pipe = fe.Pipeline(fe.load_stage_a(fold_paths, device), device,
                       scorer=fs.InstanceScorer.load(name="effv2s_scorer", tau=TAU))
else:
    pipe = fp.Pipeline(fp.load_stage_a(fold_paths, device), fp.load_refiner(refiner_path, device), device,
                       scorer=fs.InstanceScorer.load(tau=TAU))   # scorer=None: the pipeline without the instance scorer
print("device:", device, "| torch", torch.__version__, "| scorer tau", pipe.scorer.tau if pipe.scorer else None)

## 3. (Optional) Score the hold-out with the official PQ

The hold-out is the 50 images selected by `np.random.default_rng(1234).permutation` over the
sorted file names; it was excluded from every training set. Set `EVAL_HOLDOUT_IMAGES` to run it
(the first N of the 50).

In [ ]:
if EVAL_HOLDOUT_IMAGES:
    per_file, dims = fm.load_annotations(ann_json, train_dir)
    names = fm.holdout_names(list(per_file))[:EVAL_HOLDOUT_IMAGES]
    preds = {}
    for k, n in enumerate(names, 1):
        t0 = time.time()
        preds[n] = fp.label_map_to_rles(pipe.segment(fp.read_gray(os.path.join(train_dir, n))))
        print(f"{k}/{len(names)} {n}: {len(preds[n])} instances, {time.time()-t0:.0f}s", flush=True)
    res = fm.evaluate(preds, ann_json, names, train_dir)
    print("PQ", round(res["pq"], 4), {k: res[k] for k in ("tp", "fp", "fn")})
    json.dump({n: v for n, v in res["per_image"].items()},
              open(os.path.join(OUT_DIR, "holdout_eval.json"), "w"), indent=1)

## 4. Predict the test set and write `submission.csv`

In [ ]:
test_files = sorted(f for f in os.listdir(test_dir) if f.lower().endswith((".jpeg", ".jpg")))
if MAX_TEST_IMAGES:
    test_files = test_files[:MAX_TEST_IMAGES]
rows, t_start = [], time.time()
for k, fname in enumerate(test_files, 1):
    stem = fname.rsplit(".", 1)[0]
    rles = fp.label_map_to_rles(pipe.segment(fp.read_gray(os.path.join(test_dir, fname))))
    rows += [{"filament_id": f"{stem}_{i}", "segmentation_rle": r} for i, r in enumerate(rles, 1)]
    if k % 10 == 0 or k == len(test_files):
        print(f"{k}/{len(test_files)}  {time.time()-t_start:.0f}s  rows={len(rows)}", flush=True)
sub = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
sub.to_csv(os.path.join(OUT_DIR, "submission.csv"), index=False)
print(fp.validate_submission_rows(rows, {f.rsplit(".", 1)[0] for f in test_files}))

## Notes
* Images without any detected filament produce no rows (the metric counts their ground-truth
  filaments as false negatives).
* The submitted maps were computed on a GPU (EfficientNetV2-S: float16 autocast). CPU and GPU can
  differ by tiny floating-point amounts; a run on the other device may move a few boundary pixels or
  one borderline instance.
* `analysis/scripts/check_effv2s_e2e.py` compares this code path with the submitted file image by image.
* Rules: the model uses only the provided H-alpha test images at inference and only the
  competition's own annotations for training.